In [ ]:
!pip install pythainlp gensim matplotlib --quiet

In [ ]:
RANDOM_STATE = 42
BATCH_SIZE = 128
LR = 2e-4
EPOCHS = 100

HIDDEN_DIM = 300
MAX_LEN = 512
MAX_VOCAB = 20000

PAD_ID = 0
UNK_ID = 1

In [ ]:
labels = [
    "politics", "human_rights", "quality_of_life", "international",
    "social", "environment", "economics", "culture", "labor",
    "national_security", "ict", "education"
]

id_to_label = {i:l for i,l in enumerate(labels)}
NUM_CLASSES = len(labels)

In [ ]:
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, classification_report
from collections import Counter

from pythainlp.tokenize import word_tokenize
from pythainlp.util import normalize
from pythainlp.word_vector import WordVector

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
path = "/kaggle/input/pythainlp-prachatai-67k"

In [ ]:
train_df = pd.read_csv(f"{path}/train.csv")[['body_text','label']].dropna()
val_df   = pd.read_csv(f"{path}/validation.csv")[['body_text','label']].dropna()
test_df  = pd.read_csv(f"{path}/test.csv")[['body_text','label']].dropna()

In [ ]:
train_df['label_name'] = train_df['label'].map(id_to_label)
val_df['label_name']   = val_df['label'].map(id_to_label)
test_df['label_name']  = test_df['label'].map(id_to_label)

print("🔍 Label mapping check:")
print(train_df[['label','label_name']].head())

In [ ]:
def process_th(text):
    text = normalize(str(text))
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def tokenize_th(text):
    tokens = word_tokenize(text, engine="newmm", keep_whitespace=False)
    return [t for t in tokens if len(t.strip()) > 1]

train_df['text'] = train_df['body_text'].apply(process_th).apply(tokenize_th)
val_df['text']   = val_df['body_text'].apply(process_th).apply(tokenize_th)
test_df['text']  = test_df['body_text'].apply(process_th).apply(tokenize_th)

In [ ]:
counter = Counter()
for t in train_df['text']:
    counter.update(t)

vocab = {"<pad>": PAD_ID, "<unk>": UNK_ID}
for i, (w, _) in enumerate(counter.most_common(MAX_VOCAB - 2), start=2):
    vocab[w] = i

def encode(tokens):
    if len(tokens) > MAX_LEN:
        half = MAX_LEN // 2
        tokens = tokens[:half] + tokens[-half:]
    ids = [vocab.get(t, UNK_ID) for t in tokens]
    return ids + [PAD_ID] * (MAX_LEN - len(ids))

vocab_size = len(vocab)

In [ ]:
print("Loading Thai2Vec...")
thai2vec = WordVector()

In [ ]:
embedding_weight = np.zeros((vocab_size, HIDDEN_DIM), dtype=np.float32)
rng = np.random.default_rng(RANDOM_STATE)
embedding_weight[UNK_ID] = rng.normal(0, 0.01, size=(HIDDEN_DIM,))

for word, idx in vocab.items():
    if word in ("<pad>", "<unk>"):
        continue
    try:
        embedding_weight[idx] = thai2vec.get_vector(word)
    except:
        embedding_weight[idx] = rng.normal(0, 0.01, size=(HIDDEN_DIM,))

In [ ]:
class TextDataset(Dataset):
    def __init__(self, df):
        self.texts = df['text'].tolist()
        self.labels = df['label'].values

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return (
            torch.tensor(encode(self.texts[idx]), dtype=torch.long),
            torch.tensor(int(self.labels[idx]), dtype=torch.long)
        )

train_loader = DataLoader(TextDataset(train_df), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(TextDataset(val_df), batch_size=BATCH_SIZE)
test_loader  = DataLoader(TextDataset(test_df), batch_size=BATCH_SIZE)

In [ ]:
class RNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, output_dim, embedding_matrix=None):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_ID)

        if embedding_matrix is not None:
            self.embedding.weight.data.copy_(torch.tensor(embedding_matrix, dtype=torch.float32))

        self.embedding.weight.requires_grad = True

        self.rnn = nn.LSTM(embed_dim, embed_dim // 2, batch_first=True)

        self.fc1 = nn.Linear(embed_dim // 2, embed_dim // 4)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(embed_dim // 4, output_dim)

    def forward(self, text):
        lengths = (text != PAD_ID).sum(dim=1)

        embedded = self.embedding(text)

        packed = nn.utils.rnn.pack_padded_sequence(
            embedded, lengths.cpu(), batch_first=True, enforce_sorted=False
        )

        _, (hidden, _) = self.rnn(packed)

        out = hidden[-1]

        out = self.fc1(out)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.fc2(out)

        return out

model = RNN(vocab_size, HIDDEN_DIM, NUM_CLASSES, embedding_weight).to(DEVICE)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss()

In [ ]:
@torch.no_grad()
def evaluate(loader):
    model.eval()
    preds, labels_ = [], []

    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        out = model(x).argmax(1)
        preds.append(out.cpu().numpy())
        labels_.append(y.cpu().numpy())

    y_pred = np.concatenate(preds)
    y_true = np.concatenate(labels_)

    acc = (y_pred == y_true).mean()
    f1_micro = f1_score(y_true, y_pred, average="micro")
    f1_macro = f1_score(y_true, y_pred, average="macro")
    f1_weighted = f1_score(y_true, y_pred, average="weighted")

    return acc, f1_micro, f1_macro, f1_weighted, y_true, y_pred


for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for x, y in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)

        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    val_acc, f1_micro, f1_macro, f1_weighted, _, _ = evaluate(val_loader)

    print(
        f"Epoch {epoch+1:03d} | "
        f"loss={total_loss:.6f} | "
        f"acc={val_acc:.6f} ({val_acc*100:.2f}%) | "
        f"f1_micro={f1_micro:.6f} | "
        f"f1_macro={f1_macro:.6f} | "
        f"f1_weighted={f1_weighted:.6f}"
    )


In [ ]:
test_acc, f1_micro, f1_macro, f1_weighted, y_true, y_pred = evaluate(test_loader)

print(
    f"\nFINAL MODEL TEST:\n"
    f"Accuracy     = {test_acc:.6f} ({test_acc*100:.2f}%)\n"
    f"F1-micro     = {f1_micro:.6f}\n"
    f"F1-macro     = {f1_macro:.6f}\n"
    f"F1-weighted  = {f1_weighted:.6f}"
)

print("\n===== CLASSIFICATION REPORT =====")
print(classification_report(y_true, y_pred, target_names=labels))

# =========================
# PREDICT
# =========================
@torch.no_grad()
def predict(text):
    tokens = tokenize_th(process_th(text))
    x = torch.tensor([encode(tokens)], dtype=torch.long).to(DEVICE)

    probs = torch.softmax(model(x), dim=1).cpu().numpy()[0]
    pred = int(np.argmax(probs))

    return {
        "label_id": pred,
        "label_name": id_to_label[pred],
        "confidence": float(probs[pred])
    }

print(predict("รัฐบาลประกาศนโยบายเศรษฐกิจใหม่"))